___
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://www.velaninfo.com/rs/wp-content/uploads/2024/11/Data-Annotation-Labeling.jpg" width="300px" height="180px" />


# <font color= #bbc28d> **Object Detection in Urban Setting** </font>
#### <font color= #2E9AFE> `Final Project - Machine Learning`</font>
- <Strong> Sofía Maldonado, Diana Valdivia, Samantha Sánchez, Isabel Valladolid & Vivienne Toledo </Strong>
- <Strong> Fecha </Strong>: 01/12/2025.

___

<p style="text-align:right;"> Image retrieved from: https://www.velaninfo.com/rs/wp-content/uploads/2024/11/Data-Annotation-Labeling.jpg</p>

# <font color= #bbc28d> **Introduction** </font>

Urban environments present some of the most challenging scenarios for **computer vision systems**, especially in applications such as autonomous driving and advanced driver-assistance systems (ADAS). Vehicles operating in these contexts must be capable of perceiving their surroundings accurately and quickly, **identifying critical elements** such as cars, pedestrians, traffic signs, and other road objects in real time. To address this need, this project focuses on developing a **object recognition system** tailored for urban street video.

In this assignment, the entire perception pipeline is implemented within a Jupyter notebook using Python, OpenCV, and pretrained deep learning detectors from the YOLO family or other comparable high-performance models. The system is designed to **process recorded footage**, perform object detection at real-time speeds, and annotate the video with labels for key road elements. Additionally, the project evaluates the system’s performance by measuring accuracy, detection quality, and latency.

----

In [7]:
# === IMPORTS ===

# Generales
import numpy as np
import pandas as pd

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Data Processing
import json
import os
import random
import shutil
from tqdm import tqdm
from PIL import Image

# <font color= #bbc28d> **Data Loading** </font>
For training our model, we considered three options. We ultimately selected the first one: using the **BDD100K dataset** from Berkeley to train our models. However, we did not use the entire dataset. Instead, we created a curated subset. To achieve this, we used only the **validation split** (approximately 10k images) and randomly selected **2,500 images** — using a fixed seed for reproducibility — to train our models.

The validation images and annotations can be downloaded from here: https://www.kaggle.com/datasets/marquis03/bdd100k?resource=download-directory&select=val 

Before diving into the process, let's establish the general aspects of the project:

In [8]:
 # ========== CONFIGURACIÓN GENERAL ==========

# Número total de imágenes que quieres tomar de BDD100K
num_images = 2500

# Porcentajes de división del dataset
train_ratio = 0.75
val_ratio = 0.15
test_ratio = 0.10

# Rutas de las imágenes
images_dir = r"C:\Users\denis\Documents\Object_det\val\images"

# JSON de las anotaciones
labels_json = r"C:\Users\denis\Documents\Object_det\val\annotations\bdd100k_labels_images_val.json"

# Carpeta final 
out_dir = r"C:\Users\denis\Documents\Object_det\dataset_yolo"
os.makedirs(out_dir, exist_ok=True)

## <font color= #66b0b0> • **Class Mapping** </font>
This section defines the list of classes that YOLO will detect in our project. The list yolo_classes contains the seven categories we want to recognize: cars, trucks, buses, pedestrians, cyclists, traffic lights, and traffic signs. Then, the dictionary bdd_to_yolo is used to map the original BDD100K class names to the numeric IDs required by YOLO.

In [9]:
# ========== MAPEO DE CLASES PARA YOLO ==========

# Lista de clases de Yolo
yolo_classes = [
    "car", "truck", "bus",
    "pedestrian", "cyclist",
    "traffic light", "traffic sign"
]

# Mapeo del nombre de categoría 
bdd_to_yolo = {
    "car": 0,
    "truck": 1,
    "bus": 2,
    "pedestrian": 3,
    "rider": 4,           # cyclist
    "traffic light": 5,
    "traffic sign": 6
}

## <font color= #66b0b0> • **Class Mapping** </font>
This section contains the helper functions used to load, process, and convert the BDD100K annotations into the YOLO format.

##### **`load_annotations()`**
This function reads the JSON file that contains all BDD100K labels. Each entry in the JSON includes the image name and a list of annotated objects. The function simply opens the file and returns its content as a Python dictionary.

##### **`yolo_bbox()`**
BDD100K bounding boxes use the format (x1, y1, x2, y2), where the coordinates represent the top-left and bottom-right corners of the box. YOLO requires a different format: the center of the box (xc, yc) and its width and height.
This function converts each BDD100K box to the normalized YOLO format.

##### **`process_labels()`**
This function processes the entire annotation JSON and builds a dictionary where each key is an image name and each value is a list of YOLO-formatted labels. It only keeps images that:

- Actually exist in the images folder,
- Contain at least one object,
- Have valid categories from our YOLO class list.

For every object, it checks if the category is allowed, converts the bounding box with yolo_bbox(), and saves it. Only images with valid annotations are included in the final output.

##### **`write_yolo_label()`**
This function creates a .txt file for each image in YOLO format.All values are normalized and written with six decimal places, which is the format expected by YOLO.

In [10]:
# ========== FUNCIONES AUXILIARES ==========

def load_annotations(json_path):
    """
    Carga el archivo JSON de anotaciones BDD100K.
    Cada elemento del JSON contiene:
        - name (nombre de la imagen)
        - labels (lista de objetos con su bbox)
    """
    print(f"→ Cargando labels de {json_path}...")
    with open(json_path, 'r') as f:
        return json.load(f)


def yolo_bbox(box, w, h):
    """
    Convierte una caja en formato BDD100K a formato YOLO.

    BDD100K usa:
        x1, y1 = esquina superior izquierda
        x2, y2 = esquina inferior derecha

    YOLO usa:
        xc, yc = centro (normalizado)
        bw, bh = ancho y alto (normalizados)
    """
    x1, y1, x2, y2 = box
    xc = ((x1 + x2) / 2) / w
    yc = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return xc, yc, bw, bh


def process_labels(ann_data):
    """
    Procesa el JSON de anotaciones completo y crea un diccionario:

        { "nombre_imagen.jpg": [(class_id, bbox), ...] }

    SOLO incluye imágenes que:
        ✓ existen físicamente en tu carpeta
        ✓ tienen al menos un objeto etiquetado
        ✓ contienen categorías válidas según tu lista YOLO
    """
    out = {}

    for item in ann_data:
        filename = item["name"]

        # Saltamos imágenes sin labels
        if "labels" not in item:
            continue

        # Usamos PIL para obtener el tamaño real de la imagen
        img_path = os.path.join(images_dir, filename)
        if not os.path.exists(img_path):
            continue  # el JSON tiene algunas imágenes que no existen

        img = Image.open(img_path)
        w, h = img.size  # dimensiones reales de la imagen

        file_labels = []

        # Revisar todos los objetos anotados en esta imagen
        for obj in item["labels"]:

            # Solo objetos con caja 2D válida
            if "box2d" not in obj:
                continue

            cat = obj["category"]

            # Si la categoría NO está en nuestra lista, se ignora
            if cat not in bdd_to_yolo:
                continue

            cls_id = bdd_to_yolo[cat]
            b = obj["box2d"]

            # Conversión a formato YOLO
            box_list = [b["x1"], b["y1"], b["x2"], b["y2"]]

            bbox = yolo_bbox(box_list, w, h)

            file_labels.append((cls_id, bbox))

        # Solo guardamos imágenes que tengan objetos válidos
        if file_labels:
            out[filename] = file_labels

    return out

def write_yolo_label(path, labels):
    """
    Crea un archivo .txt con las anotaciones en formato YOLO.
    Cada línea tiene:
        <class> <xc> <yc> <bw> <bh>
    """
    with open(path, "w") as f:
        for cls_id, (xc, yc, bw, bh) in labels:
            f.write(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

## <font color= #66b0b0> • **Dataset Creation** </font>

This block loads the annotations, filters the valid ones, and builds the final YOLO dataset.

**1. Load annotations**: We read the JSON file and extract all objects and bounding boxes. Only images that exist and contain valid categories are kept.

**2. Select 2,500 images**: From all valid images, we randomly choose 2500, a fixed seed is used so the selection is always the same.

**3. Train/Val/Test split**: The 2,500 images are split into:
- **Train:** 75%**
- **Val:** 15%**
- **Test:** 10%

**4. Create YOLO folder structure**: We create the standard YOLO directories with the images and annotations.

**5. Copy images and save labels**: For each image:
- The image is copied into the correct folder.
- A `.txt` file is created with its YOLO annotations.


In [11]:
# ========== EJECUTAR Y JUNTAR ==========

# Fijar semilla para reproducibilidad
random.seed(42)

print("Cargando anotaciones del JSON...")
ann_data = load_annotations(labels_json)

print("Procesando y filtrando labels válidos...")
all_labels = process_labels(ann_data)

print(f"✔ Total imágenes con anotaciones válidas: {len(all_labels)}")

# Selección aleatoria de 2500 imágenes
selected_imgs = random.sample(list(all_labels.keys()), num_images)

# División en train/val/test
n_train = int(num_images * train_ratio)
n_val = int(num_images * val_ratio)

train_set = selected_imgs[:n_train]
val_set = selected_imgs[n_train:n_train + n_val]
test_set = selected_imgs[n_train + n_val:]

splits = {
    "train": train_set,
    "val": val_set,
    "test": test_set
}

# Crear toda la estructura de carpetas YOLO
for split in splits:
    os.makedirs(f"{out_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{out_dir}/labels/{split}", exist_ok=True)

print("\nCopiando imágenes y generando archivos YOLO...")

# Copia de imágenes + creación de .txt
for split, imgs in splits.items():
    for img in tqdm(imgs, desc=f"Procesando {split}"):
        
        # Ruta origen y destino de la imagen
        src = os.path.join(images_dir, img)
        dst = os.path.join(out_dir, "images", split, img)
        shutil.copy(src, dst)

        # Guardar archivo .txt con bounding boxes
        lbl_path = os.path.join(out_dir, "labels", split, img.replace(".jpg", ".txt"))
        write_yolo_label(lbl_path, all_labels[img])

print("\nDataset YOLO generado correctamente\n")

Cargando anotaciones del JSON...
→ Cargando labels de C:\Users\denis\Documents\Object_det\val\annotations\bdd100k_labels_images_val.json...
Procesando y filtrando labels válidos...
✔ Total imágenes con anotaciones válidas: 9992

Copiando imágenes y generando archivos YOLO...


Procesando test: 100%|██████████| 250/250 [00:00<00:00, 914.20it/s]


Dataset YOLO generado correctamente



## <font color= #66b0b0> • **YAML Creation** </font>

This section generates the `dataset.yaml` file required by YOLO. The YAML tells the model where the images and labels are located and what classes it should expect:
- The paths to the **train**, **val**, and **test** image folders.
- The number of classes (`nc`).
- The list of class names (`names`).

YOLO uses this file during training to correctly load the dataset and map the class IDs to their names.

In [12]:
# ========== CREAR UN YAML PARA YOLO ==========
yaml_path = os.path.join(out_dir, "dataset.yaml")

with open(yaml_path, "w") as f:
    f.write(f"train: {out_dir}/images/train\n")
    f.write(f"val: {out_dir}/images/val\n")
    f.write(f"test: {out_dir}/images/test\n\n")
    f.write(f"nc: {len(yolo_classes)}\n")
    f.write(f"names: {yolo_classes}\n")

print(f"dataset.yaml generado en: {yaml_path}")

dataset.yaml generado en: C:\Users\denis\Documents\Object_det\dataset_yolo\dataset.yaml


# <font color= #bbc28d> **MODELING** </font>

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print('ya')
model.train(
    data=r"C:\Users\denis\Documents\Object_det\dataset_yolo\dataset.yaml",
    epochs=1,
    imgsz=250,
    batch=16
)